# Reading a protein into mBuild and writing a new PDB file
### Joseph R. Laforet Jr.

A PDB file gives you elements, atom names, and coordinates. You also need the bond orders and the formal
charge of every atom to assign force
field parameters. Those are not always present in PDB files.

Some PDB readers guess this information from interatomic
distances, or omit charges entirely. 

`mbuild.biopolymers` does not guess chemistry. Every residue is matched by atom
name against a template from the wwPDB Chemical Component Dictionary,
and the bonds, bond orders and formal charges come from that template. A
residue not given by a template raises an error.

This notebook teaches three things about the module:

- A loaded protein is a tree of mBuild Compounds: `Protein(mb.Compound)` → `Chain(mb.Compound)` →
  `Residue(mb.Compound)` → atoms. Everything mBuild does with a Compound works on each level.
- The chemistry of a residue lives on its `template`, a `ResidueTemplate`
  matched from the CCD. The formal charges are copied from it.
- Export is a check. `to_rdkit` and `save_pdb` only succeed when the
  chemistry survived the load.

In the review stack these are PRs 1 to 4: residue definitions, matching,
the Compound hierarchy, and writing back out.

In [ ]:
from mbuild.biopolymers import Protein

mbuild_protein = Protein("1ubq_protonated.pdb")
print(len(list(mbuild_protein.residues())), "residues,", mbuild_protein.n_particles, "atoms")
print("net formal charge:", mbuild_protein.net_formal_charge)

## The tree

`Protein`, `Chain` and `Residue` are subclasses of `mb.Compound`, so the
usual `children`, `particles()`, `bonds()` and `clone` all apply. What the
subclasses add is the bookkeeping a PDB file carries and a plain Compound
does not: `chain_id` on a chain, `resnum`, `icode` and `hetatm` on a residue.
`get_residue` and `get_atom` address them the way the file does, by residue
number and chain.

In [ ]:
import mbuild as mb

chain = list(mbuild_protein.chains)[0]
residue = chain.children[0]
atom = next(residue.particles())

for obj in (mbuild_protein, chain, residue, atom):
    print(f"{type(obj).__name__:10s} {obj.name:10s} is an mb.Compound: {isinstance(obj, mb.Compound)}")
    
print(f"chain {chain.chain_id} holds {len(chain.children)} residues; "
      f"residue {residue.name} {residue.resnum} holds {residue.n_particles} atoms")

## Where the chemistry lives

Formal charges of residues are not something a coordinate reader gives you. They come
from the matched CCD templates, one residue at a time.

Each `Residue` keeps the template it matched as `residue.template`. 

A `ResidueTemplate` is one protonation and termination variant of a CCD
component: its atom names, elements, bonds with orders, and per-atom formal
charges. 

The CCD holds several variants per residue, and the matcher
picks the one whose atoms are present in the input file. `formal_charge` and
`atom_formal_charges` on the residue are copied from that variant, so the
three always agree. 

A residue can also have `template = None`; notebooks 3
and 4 make such residues on purpose to showcase how to work with custom amino acid residues.

In [ ]:
for resnum in (1, 48, 76):
    residue = mbuild_protein.get_residue(resnum, chain_id="A")
    print(f"{residue.name} {resnum:>3}  charge {residue.formal_charge:+d}  "
          f"{residue.atom_formal_charges}")

In [ ]:
from mbuild.biopolymers import CCDLibrary

template = mbuild_protein.get_residue(48, chain_id="A").template
print(type(template).__name__, template.name, "|", template.description)
print("atoms this variant expects:", sorted(template.atom_names))
print("variants of LYS in the bundled library:", len(CCDLibrary()["LYS"]))

Bond order is preserved! `to_rdkit` allows you to see that: it
refuses to export a bond whose order is unknown, so the fact that it
returns a sanitized molecule at all is the check.

That is the pattern for every export in the module: the exporter refuses
rather than guesses, so a successful export is evidence about the load.

In [ ]:
from rdkit import Chem
from collections import Counter

rdkit_mol = mbuild_protein.to_rdkit()
print(rdkit_mol.GetNumAtoms(), "atoms, formal charge", Chem.GetFormalCharge(rdkit_mol))

rdkit_mol_bond_orders = Counter(bond.GetBondType() for bond in rdkit_mol.GetBonds())
print(rdkit_mol_bond_orders)

## Handing it to OpenFF

A bond is listed as a CONECT record only when
residue order does not imply it. Peptide bonds are implied, so they are left
out. Disulfides and, later, attached fragments are not, so they are written.

`save_pdb` writes a file with real residue numbers, chain identifiers,
a TER after each chain, and CONECT records only for the bonds that
residue order does not imply, such as disulfides. A residue-template reader rejects a CONECT its own
definitions cannot explain, which is why the peptide bonds must stay implied.

Ubiquitin has no cysteines, so this file has no CONECT records at all. We are using Ubiquitin for the covalent modification demo later, though.
The lysozyme round trip in
[`evidence/pdb_round_trip.ipynb`](evidence/pdb_round_trip.ipynb) shows
the four disulfides written as CONECT records and read back.

openff-pablo reads that file with no extra arguments.


In [ ]:
from openff.pablo import STD_CCD_CACHE, topology_from_pdb

mbuild_protein.save_pdb("1ubq_FROM_MBUILD.pdb", overwrite=True)

# Note, we can load the mBuild generated protein into OpenFF via Pablo
openff_topology = topology_from_pdb("1ubq_FROM_MBUILD.pdb", residue_library=STD_CCD_CACHE)

openff_molecule = openff_topology.molecule(0)
print(openff_molecule.n_atoms, "atoms, net charge", openff_molecule.total_charge)

openff_molecule_bond_orders = Counter(bond.bond_order for bond in openff_molecule.bonds)
print("RDKit Bond Orders (not kekulized): ", rdkit_mol_bond_orders)
print("OpenFF Bond Orders (kekulized): ", openff_molecule_bond_orders, f"Aromatic: {sum(b.is_aromatic for b in openff_molecule.bonds)}")

print("NOTE: The 5 missing aromatic bonds are from a neutral imidazole in histidine that are not labeled as aromatic by OpenFF's aromaticity model.")

In [ ]:
view = openff_topology.visualize()
view.clear_representations()
view.add_representation("cartoon", color="#990000")
view

That is the round trip for an unmodified protein. To take away:

- The tree is Compounds all the way down, with PDB bookkeeping added.
- Chemistry comes from `residue.template`, never from distances.
- An export that returns is a check that passed.

The next notebook modifies the protein and hands it to OpenFF.